# Grad-CAM Analysis Per Task Head

Grad-CAM computed separately for the species and freshness heads of the best multi-task model, to check whether attention concentrates on the eye region or leans on background.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
FIGURES_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results/figures/gradcam'

import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
!unzip -q "$DATASET_ZIP" -d /content/data

In [ ]:
import torch
from evaluate import load_multitask_model

BEST_RUN_NAME = 'ModelC_UW_seed44'  # set from the ranking in notebook 05
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = load_multitask_model(f'{CHECKPOINT_DIR}/{BEST_RUN_NAME}.pt', DEVICE)

In [ ]:
import pandas as pd

test_df = pd.read_csv('/content/repo/02_Manifests/test.csv')

# All 3 freshness levels, not just the extremes -- the background-reliance risk in
# 3.3.9 applies across the whole range, and the middle "Fresh" condition is the one
# that tests whether attention shifts smoothly (ordinal) rather than erratically.
sample_df = (
    test_df.groupby(['species', 'freshness'])
    .head(1)
    .reset_index(drop=True)
)
len(sample_df)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from dataset import eval_transform
from gradcam_utils import compute_gradcam, denormalize_for_display

for _, row in sample_df.iterrows():
    image_path = os.path.join(DATASET_ROOT, row['filepath'])
    pil_image = Image.open(image_path).convert('RGB')
    img_tensor = eval_transform(pil_image)
    rgb_float = denormalize_for_display(img_tensor)

    cam_species = compute_gradcam(model, img_tensor, rgb_float, head='species')
    cam_freshness = compute_gradcam(model, img_tensor, rgb_float, head='freshness')

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb_float); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cam_species); axes[1].set_title('Species Head'); axes[1].axis('off')
    axes[2].imshow(cam_freshness); axes[2].set_title('Freshness Head'); axes[2].axis('off')
    fig.suptitle(f"{row['species']} - {row['freshness']}")
    plt.tight_layout()

    safe_name = f"{row['species']}_{row['freshness']}".replace(' ', '_')
    fig.savefig(os.path.join(FIGURES_DIR, f'{safe_name}.png'), dpi=150)
    plt.show()
    plt.close(fig)

Figures are saved to `FIGURES_DIR` on Drive, not to `/content/repo`. To add them to the GitHub repository, download the selected images from Drive and commit them from a machine with push access.